# RL Evaluation & Visualisation

This notebook evaluates and visualises the training outcomes for three RL approaches
on the **GridWorld** environment:

| # | Approach | Reference |
|---|----------|-----------|
| 1 | **Random Baseline** | — |
| 2 | **GCRL** — Single-Goal Contrastive RL | Liu, Tang & Eysenbach (2024) |
| 3 | **RCRL** — Reward-Conditioned Q-Learning | Nauman, Cygan & Abbeel (2026) |

**Sections**
1. [Setup & Run / Load Experiments](#section-1)
2. [Reward over Time](#section-2)
3. [Epsilon (ε) Decay over Training](#section-3)
4. [Agent Trajectory Visualisation](#section-4)

---
> **Tip:** If you have not run the experiments yet, executing the *Setup* cell will
> train all three approaches automatically (takes ≈ 1–2 min on a 5×5 grid).

## Imports

In [ ]:
from __future__ import annotations

import sys
import os
import warnings

# Add the repository root to sys.path so project modules can be imported
sys.path.insert(0, os.path.abspath('..'))
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

# Consistent, presentation-ready style
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
})

print('Imports OK')

<a id='section-1'></a>
## Section 1 — Setup & Run / Load Experiments

Edit the constants below to match the grid size and experiment configuration you
want to analyse.  The cell then checks whether log files already exist; if they
do not, it runs all three approaches via `compare_all()` and saves the results.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Configuration — edit these to match your experiment settings
# ─────────────────────────────────────────────────────────────────────────────

LOG_DIR      = Path('../logs/compare')   # must match --log-dir in compare_approaches.py
GRID_HEIGHT  = 5
GRID_WIDTH   = 5
N_EPISODES   = 300   # training episodes per approach
MAX_STEPS    = 200   # max steps per episode
SEED         = 42

# Derived constants
GOAL_STATE   = GRID_HEIGHT * GRID_WIDTH - 1   # bottom-right cell
START_STATE  = 0                               # top-left cell

APPROACHES = ['random', 'gcrl', 'rcrl']
LABELS = {
    'random': 'Random Baseline',
    'gcrl':   'GCRL (Contrastive RL)',
    'rcrl':   'RCRL (Reward-Conditioned)',
}
COLORS = {
    'random': '#e74c3c',
    'gcrl':   '#2ecc71',
    'rcrl':   '#3498db',
}

# ─────────────────────────────────────────────────────────────────────────────
# Run experiments
# Set FORCE_RERUN = True (default) to always re-run and overwrite existing logs.
# Set to False to reuse previously saved CSVs without re-running.
# ─────────────────────────────────────────────────────────────────────────────

FORCE_RERUN = True   # always overwrite logs with fresh results

def _logs_exist() -> bool:
    return all((LOG_DIR / ap / 'metrics.csv').exists() for ap in APPROACHES)


if FORCE_RERUN or not _logs_exist():
    print('Running experiments (logs will be overwritten) …')
    print('(This may take a couple of minutes on a 5×5 grid.)\n')

    from experiments.compare_approaches import compare_all, parse_args

    args = parse_args([
        '--height',      str(GRID_HEIGHT),
        '--width',       str(GRID_WIDTH),
        '--episodes',    str(N_EPISODES),
        '--max-steps',   str(MAX_STEPS),
        '--seed',        str(SEED),
        '--log-dir',     str(LOG_DIR),
        '--goal-state',  str(GOAL_STATE),
        '--start-state', str(START_STATE),
    ])
    _results = compare_all(args)
    print('\nExperiments complete!')
else:
    print(f'Existing logs found in {LOG_DIR} — loading without re-running …')

### Comparison Table

Summarise the key metrics for each approach from the saved CSVs.

In [ ]:
# Load all metrics CSVs
dfs: dict[str, pd.DataFrame] = {}
for ap in APPROACHES:
    path = LOG_DIR / ap / 'metrics.csv'
    if path.exists():
        dfs[ap] = pd.read_csv(path)
    else:
        print(f'WARNING: {path} not found — skipping {ap}')


def _safe_mean(series: pd.Series) -> str:
    return f'{series.mean():.4f}' if len(series) > 0 else 'N/A'


rows = []
for ap in APPROACHES:
    if ap not in dfs:
        continue
    df   = dfs[ap]
    tr   = df[df['mode'] == 'train']['total_reward']
    ev   = df[df['mode'] == 'eval']['total_reward']
    k    = max(1, len(tr) // 10)
    last = tr.iloc[-k:]
    rows.append({
        'Approach':          LABELS[ap],
        'Mean Train Reward': _safe_mean(tr),
        'Last-10% Reward':   _safe_mean(last),
        'Mean Eval Reward':  _safe_mean(ev) if len(ev) > 0 else 'N/A',
        'Train Episodes':    len(tr),
        'Eval Episodes':     len(ev),
    })

summary = pd.DataFrame(rows).set_index('Approach')
print('COMPARISON TABLE')
print('=' * 72)
display(summary)

<a id='section-2'></a>
## Section 2 — Reward over Time

Each subplot shows:
- **Faint line** — raw episode reward
- **Solid line** — rolling mean (window = `SMOOTH`)
- **Shaded band** — rolling ±1 standard deviation
- **◆ markers** — evaluation / exploitation episodes

The overlay plot shows smoothed rewards for all three approaches on the same axes
for direct comparison.

In [ ]:
SMOOTH = 20   # rolling-mean window width

fig, axes = plt.subplots(1, len(APPROACHES), figsize=(14, 4), sharey=False)
fig.suptitle('Episode Reward over Training', fontsize=14, fontweight='bold', y=1.02)

for ax, ap in zip(axes, APPROACHES):
    if ap not in dfs:
        ax.set_visible(False)
        continue

    df    = dfs[ap]
    train = df[df['mode'] == 'train'].reset_index(drop=True)
    evdf  = df[df['mode'] == 'eval'].reset_index(drop=True)

    ep  = train['episode']
    rew = train['total_reward']
    mu  = rew.rolling(SMOOTH, min_periods=1).mean()
    sd  = rew.rolling(SMOOTH, min_periods=1).std(ddof=0).fillna(0)

    # Raw rewards (faint background)
    ax.plot(ep, rew, alpha=0.20, color=COLORS[ap], linewidth=0.8)
    # Rolling mean
    ax.plot(ep, mu, color=COLORS[ap], linewidth=2.2, label=f'Rolling mean (w={SMOOTH})')
    # ±1 std band
    ax.fill_between(ep, mu - sd, mu + sd, alpha=0.15, color=COLORS[ap])

    # Evaluation / exploitation checkpoints
    if len(evdf) > 0:
        ax.scatter(
            evdf['episode'], evdf['total_reward'],
            marker='D', s=35, color='black', zorder=5, label='Eval / exploit',
        )

    ax.set_title(LABELS[ap], fontsize=11, fontweight='bold')
    ax.set_xlabel('Episode')
    ax.set_ylabel('Total Reward')
    ax.legend(fontsize=8)

plt.tight_layout()
LOG_DIR.mkdir(parents=True, exist_ok=True)
plt.savefig(LOG_DIR / 'reward_over_time_subplots.png', bbox_inches='tight')
plt.show()
print(f'Saved → {LOG_DIR / "reward_over_time_subplots.png"}')

In [ ]:
# ─── Overlay comparison ──────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
ax.set_title('Smoothed Reward — All Approaches Overlaid', fontsize=13, fontweight='bold')

for ap in APPROACHES:
    if ap not in dfs:
        continue
    train = dfs[ap][dfs[ap]['mode'] == 'train'].reset_index(drop=True)
    ep    = train['episode']
    mu    = train['total_reward'].rolling(SMOOTH, min_periods=1).mean()
    sd    = train['total_reward'].rolling(SMOOTH, min_periods=1).std(ddof=0).fillna(0)

    ax.plot(ep, mu, color=COLORS[ap], linewidth=2.5, label=LABELS[ap])
    ax.fill_between(ep, mu - sd, mu + sd, alpha=0.12, color=COLORS[ap])

ax.set_xlabel('Training Episode')
ax.set_ylabel(f'Total Reward (rolling mean, w={SMOOTH})')
ax.legend()
plt.tight_layout()
plt.savefig(LOG_DIR / 'reward_over_time_overlay.png', bbox_inches='tight')
plt.show()
print(f'Saved → {LOG_DIR / "reward_over_time_overlay.png"}')

<a id='section-3'></a>
## Section 3 — Epsilon (ε) Decay over Training

- **Random Baseline**: ε = 1.0 throughout (always random, no learning).
- **RCRL**: ε is decayed multiplicatively each episode; values are read from the
  `epsilon` column of the CSV (logged by `Logger`).  If the column is absent the
  curve is re-computed analytically from the default hyperparameters.
- **GCRL**: does not use ε-greedy — it uses a softmax temperature τ instead.
  This plot therefore omits GCRL and notes the difference.

In [ ]:
# Default RCRL hyperparameters (used only when epsilon column is absent)
EPS_INIT  = 1.0
EPS_MIN   = 0.05
EPS_DECAY = 0.995

fig, ax = plt.subplots(figsize=(9, 4))
ax.set_title('Exploration Schedule (ε) over Training', fontsize=13, fontweight='bold')

plotted_any = False

for ap in APPROACHES:
    if ap not in dfs:
        continue

    train = dfs[ap][dfs[ap]['mode'] == 'train'].reset_index(drop=True)

    if ap == 'random':
        ax.plot(
            train['episode'], np.ones(len(train)),
            color=COLORS[ap], linewidth=2, linestyle='--',
            label=f"{LABELS[ap]}  (ε = 1.0, always random)",
        )
        plotted_any = True

    elif ap == 'rcrl':
        eps_col = pd.to_numeric(train.get('epsilon', pd.Series(dtype=float)), errors='coerce')
        valid   = eps_col.notna() & (eps_col > 0)
        if valid.any():
            ax.plot(
                train['episode'][valid], eps_col[valid],
                color=COLORS[ap], linewidth=2,
                label=f"{LABELS[ap]}  (logged ε)",
            )
        else:
            # Fall back to analytic computation
            n   = len(train)
            eps = np.maximum(EPS_MIN, EPS_INIT * EPS_DECAY ** np.arange(n))
            ax.plot(
                train['episode'], eps,
                color=COLORS[ap], linewidth=2,
                label=f"{LABELS[ap]}  (analytic: ε₀={EPS_INIT}, decay={EPS_DECAY})",
            )
        plotted_any = True

    # gcrl: no epsilon — handled by the annotation below

if plotted_any:
    ax.set_xlabel('Training Episode')
    ax.set_ylabel('Epsilon (ε)')
    ax.set_ylim(-0.05, 1.15)
    ax.legend()
    ax.annotate(
        'GCRL uses softmax temperature τ instead of ε — not shown here.',
        xy=(0.02, 0.06), xycoords='axes fraction',
        fontsize=9, color='grey', fontstyle='italic',
    )
    plt.tight_layout()
    plt.savefig(LOG_DIR / 'epsilon_decay.png', bbox_inches='tight')
    plt.show()
    print(f'Saved → {LOG_DIR / "epsilon_decay.png"}')
else:
    plt.close()
    print('No data available for epsilon plot.')

<a id='section-4'></a>
## Section 4 — Agent Trajectory Visualisation

Each panel shows the **last evaluation / exploitation episode** for one approach:

- **Background heatmap** — per-cell visit count (blue intensity)
- **Blue arrows** — step-by-step path taken by the agent
- **🟢 Green dot** — start position
- **⭐ Red star** — goal position

Repeated transitions are drawn once (with a thicker arrow) to keep the plot readable.

In [ ]:
# Load trajectory CSVs
traj_dfs: dict[str, pd.DataFrame] = {}
for ap in APPROACHES:
    path = LOG_DIR / ap / 'trajectory.csv'
    if path.exists():
        traj_dfs[ap] = pd.read_csv(path)
        print(f'Loaded trajectory for {LABELS[ap]}  ({len(traj_dfs[ap])} steps)')
    else:
        print(f'No trajectory file found for {ap}  ({path})')

In [ ]:
def plot_trajectory(
    ax,
    traj_df: pd.DataFrame,
    grid_height: int,
    grid_width: int,
    goal_state: int,
    start_state: int,
    title: str,
    color: str,
) -> None:
    """Render a single-episode trajectory on a GridWorld grid."""
    # Background: state visit counts as a light heatmap
    visit_counts = np.zeros((grid_height, grid_width), dtype=float)
    for state in traj_df['state']:
        r, c = divmod(int(state), grid_width)
        visit_counts[r, c] += 1

    ax.imshow(
        visit_counts, cmap='Blues', aspect='equal',
        vmin=0, vmax=max(1.0, visit_counts.max()),
        extent=[-0.5, grid_width - 0.5, grid_height - 0.5, -0.5],
    )

    # Grid lines
    for x in range(grid_width + 1):
        ax.axvline(x - 0.5, color='gray', linewidth=0.5, alpha=0.4)
    for y in range(grid_height + 1):
        ax.axhline(y - 0.5, color='gray', linewidth=0.5, alpha=0.4)

    # Trajectory arrows (deduplicated)
    states = traj_df['state'].tolist()
    transition_counts: dict[tuple, int] = {}
    for i in range(len(states) - 1):
        key = (states[i], states[i + 1])
        transition_counts[key] = transition_counts.get(key, 0) + 1

    max_count = max(transition_counts.values()) if transition_counts else 1
    for (s1, s2), count in transition_counts.items():
        r1, c1 = divmod(s1, grid_width)
        r2, c2 = divmod(s2, grid_width)
        if r1 == r2 and c1 == c2:
            continue   # skip zero-length moves (agent bounced off a wall)
        lw = 1.0 + 2.5 * (count / max_count)   # thicker line = more frequent
        ax.annotate(
            '', xy=(c2, r2), xytext=(c1, r1),
            arrowprops=dict(
                arrowstyle='->', color=color,
                lw=lw, mutation_scale=12,
            ),
        )

    # Start marker
    sr, sc = divmod(start_state, grid_width)
    ax.plot(
        sc, sr, 'o', color='limegreen', markersize=12,
        markeredgecolor='black', markeredgewidth=1.5,
        zorder=10, label='Start',
    )

    # Goal marker
    gr, gc = divmod(goal_state, grid_width)
    ax.plot(
        gc, gr, '*', color='red', markersize=16,
        markeredgecolor='darkred', markeredgewidth=1,
        zorder=10, label='Goal',
    )

    ax.set_xlim(-0.5, grid_width - 0.5)
    ax.set_ylim(grid_height - 0.5, -0.5)
    ax.set_xticks(range(grid_width))
    ax.set_yticks(range(grid_height))
    ax.set_xticklabels([])
    ax.set_yticklabels([])
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.legend(loc='upper right', fontsize=8)


# ─── Plot ────────────────────────────────────────────────────────────────────
available = [ap for ap in APPROACHES if ap in traj_dfs]

if not available:
    print('No trajectory data found.  Run the experiments first.')
else:
    n = len(available)
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 5))
    if n == 1:
        axes = [axes]

    goal_r, goal_c = divmod(GOAL_STATE, GRID_WIDTH)
    fig.suptitle(
        f'Agent Trajectory — Last Evaluation Episode\n'
        f'({GRID_HEIGHT}×{GRID_WIDTH} GridWorld | '
        f'start=(0,0), goal=({goal_r},{goal_c}))',
        fontsize=13, fontweight='bold',
    )

    for ax, ap in zip(axes, available):
        plot_trajectory(
            ax, traj_dfs[ap],
            GRID_HEIGHT, GRID_WIDTH,
            GOAL_STATE, START_STATE,
            title=LABELS[ap],
            color=COLORS[ap],
        )

    plt.tight_layout()
    plt.savefig(LOG_DIR / 'trajectories.png', bbox_inches='tight')
    plt.show()
    print(f'Saved → {LOG_DIR / "trajectories.png"}')

## Path Length to Goal over Training


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Path Length to Goal over Training
# Shows how the number of steps needed to reach the goal changes across
# training episodes.  Only episodes where the agent reached the goal
# (total_reward > 0) are plotted for training.  Eval episodes are always shown.
# ─────────────────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, len(APPROACHES), figsize=(14, 4), sharey=False)
fig.suptitle('Path Length to Goal over Training', fontsize=14, fontweight='bold', y=1.02)

for ax, ap in zip(axes, APPROACHES):
    if ap not in dfs:
        ax.set_visible(False)
        continue

    df     = dfs[ap]
    train  = df[(df['mode'] == 'train') & (df['total_reward'] > 0)].reset_index(drop=True)
    evdf   = df[df['mode'] == 'eval'].reset_index(drop=True)

    # Scatter training success episodes
    if len(train) > 0:
        ax.scatter(
            train['episode'], train['length'],
            alpha=0.25, s=15, color=COLORS[ap], label='Train (reached goal)',
        )
        # Rolling mean over success episodes
        mu = train['length'].rolling(min(SMOOTH, len(train)), min_periods=1).mean()
        ax.plot(train['episode'], mu, color=COLORS[ap], linewidth=2,
                label=f'Rolling mean (w={min(SMOOTH, len(train))})')

    # Eval episodes
    if len(evdf) > 0:
        eval_success = evdf[evdf['total_reward'] > 0]
        if len(eval_success) > 0:
            ax.scatter(
                eval_success['episode'], eval_success['length'],
                marker='D', s=40, color='black', zorder=5, label='Eval (reached goal)',
            )

    ax.set_title(LABELS[ap], fontsize=11, fontweight='bold')
    ax.set_xlabel('Episode')
    ax.set_ylabel('Steps to Goal')
    ax.legend(fontsize=8)

plt.tight_layout()
LOG_DIR.mkdir(parents=True, exist_ok=True)
plt.savefig(LOG_DIR / 'path_length.png', bbox_inches='tight')
plt.show()
print(f'Saved → {LOG_DIR / "path_length.png"}')


## Q-Value Heatmaps — Training Progression


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Q-Value Heatmaps — Early / Mid / Late Training Stages
#
# For RCRL : Q[state, nominal_ψ-bin, action]  →  saved as q_{stage}.npy
# For GCRL : C[state, action, eval_goal]       →  saved as q_{stage}.npy
#
# Each subplot shows:
#   • Left half: state×action Q-value matrix (with labelled axes)
#   • Right half (grid layout): V(s)=max_a Q[s,a] mapped onto the GridWorld,
#     with an arrow overlay showing the greedy action at each state.
# ─────────────────────────────────────────────────────────────────────────────

ACTION_NAMES  = ['UP', 'DOWN', 'LEFT', 'RIGHT']
ACTION_DELTAS = [(-1, 0), (1, 0), (0, -1), (0, 1)]   # (dr, dc) per action
STAGES        = ['early', 'mid', 'late']

# Colour-map shared across stages so comparisons are on the same scale
_CMAP = 'RdYlGn'

for ap in ['rcrl', 'gcrl']:
    q_files = [(s, LOG_DIR / ap / f'q_{s}.npy') for s in STAGES]
    available = [(s, f) for s, f in q_files if f.exists()]
    if not available:
        print(f'No Q-snapshots found for {ap} — skipping.')
        continue

    n_stages = len(available)
    # 2 sub-columns per stage: matrix + grid → n_stages * 2 columns total
    fig, axes = plt.subplots(
        2, n_stages,
        figsize=(6 * n_stages, 10),
        gridspec_kw={'height_ratios': [1, 1]},
    )
    if n_stages == 1:
        axes = axes.reshape(2, 1)

    fig.suptitle(
        f'{LABELS[ap]} — Q-Value Heatmaps (state × action)\n'        f'Goal state {GOAL_STATE} ({GOAL_STATE // GRID_WIDTH}, {GOAL_STATE % GRID_WIDTH})',
        fontsize=13, fontweight='bold',
    )

    # Compute global vmin/vmax across all stages for a consistent colour scale
    all_vals = np.concatenate([np.load(f).ravel() for _, f in available])
    vmin, vmax = float(all_vals.min()), float(all_vals.max())
    if vmin == vmax:
        vmin -= 0.01; vmax += 0.01

    for col_idx, (stage, fpath) in enumerate(available):
        Q = np.load(fpath)   # shape: (n_states, n_actions)

        # ── Row 0: state × action heatmap ────────────────────────────────────
        ax_mat = axes[0, col_idx]
        state_labels = [
            f's{i} ({i // GRID_WIDTH},{i % GRID_WIDTH})' for i in range(GRID_HEIGHT * GRID_WIDTH)
        ]
        im = ax_mat.imshow(Q, cmap=_CMAP, aspect='auto', vmin=vmin, vmax=vmax)
        ax_mat.set_xticks(range(4))
        ax_mat.set_xticklabels(ACTION_NAMES, fontsize=8)
        ax_mat.set_yticks(range(GRID_HEIGHT * GRID_WIDTH))
        ax_mat.set_yticklabels(state_labels, fontsize=6)
        ax_mat.set_xlabel('Action', fontsize=9)
        ax_mat.set_ylabel('State', fontsize=9)
        ax_mat.set_title(f'{stage.capitalize()} stage', fontsize=11, fontweight='bold')
        # Annotate each cell with Q-value
        for s in range(GRID_HEIGHT * GRID_WIDTH):
            for a in range(4):
                ax_mat.text(
                    a, s, f'{Q[s, a]:.2f}',
                    ha='center', va='center', fontsize=5.5, color='black',
                )
        plt.colorbar(im, ax=ax_mat, shrink=0.8, label='Q-value')

        # ── Row 1: V(s) = max_a Q[s,a] mapped onto the GridWorld grid ────────
        ax_grid = axes[1, col_idx]
        V = Q.max(axis=1).reshape(GRID_HEIGHT, GRID_WIDTH)
        best_a = Q.argmax(axis=1).reshape(GRID_HEIGHT, GRID_WIDTH)

        im2 = ax_grid.imshow(
            V, cmap=_CMAP, aspect='equal',
            vmin=vmin, vmax=vmax,
            extent=[-0.5, GRID_WIDTH - 0.5, GRID_HEIGHT - 0.5, -0.5],
        )
        # Grid lines
        for x in range(GRID_WIDTH + 1):
            ax_grid.axvline(x - 0.5, color='gray', linewidth=0.5, alpha=0.4)
        for y in range(GRID_HEIGHT + 1):
            ax_grid.axhline(y - 0.5, color='gray', linewidth=0.5, alpha=0.4)

        # Greedy-action arrows and state labels
        for r in range(GRID_HEIGHT):
            for c in range(GRID_WIDTH):
                s_idx = r * GRID_WIDTH + c
                # State index label (top-left corner of cell)
                ax_grid.text(
                    c - 0.42, r - 0.38, str(s_idx),
                    fontsize=6, color='white', fontweight='bold', alpha=0.8,
                )
                # Greedy-action arrow (skip goal state)
                if s_idx == GOAL_STATE:
                    continue
                dr, dc = ACTION_DELTAS[int(best_a[r, c])]
                ax_grid.annotate(
                    '', xy=(c + 0.3 * dc, r + 0.3 * dr), xytext=(c, r),
                    arrowprops=dict(
                        arrowstyle='->', color='white', lw=1.5, mutation_scale=10,
                    ),
                )

        # Start and Goal markers
        sr, sc = divmod(START_STATE, GRID_WIDTH)
        gr, gc = divmod(GOAL_STATE, GRID_WIDTH)
        ax_grid.plot(sc, sr, 'o', color='limegreen', markersize=10,
                     markeredgecolor='black', markeredgewidth=1.2, zorder=10)
        ax_grid.plot(gc, gr, '*', color='red', markersize=14,
                     markeredgecolor='darkred', markeredgewidth=0.8, zorder=10)

        ax_grid.set_xlim(-0.5, GRID_WIDTH - 0.5)
        ax_grid.set_ylim(GRID_HEIGHT - 0.5, -0.5)
        ax_grid.set_xticks(range(GRID_WIDTH))
        ax_grid.set_yticks(range(GRID_HEIGHT))
        ax_grid.set_xticklabels(range(GRID_WIDTH), fontsize=8)
        ax_grid.set_yticklabels(range(GRID_HEIGHT), fontsize=8)
        ax_grid.set_xlabel('Col', fontsize=9)
        ax_grid.set_ylabel('Row', fontsize=9)
        ax_grid.set_title(f'V(s) = max_a Q[s,a] — {stage.capitalize()}',
                          fontsize=10, fontweight='bold')
        plt.colorbar(im2, ax=ax_grid, shrink=0.8, label='V(s)')

    plt.tight_layout()
    out_path = LOG_DIR / f'{ap}_q_heatmap.png'
    plt.savefig(out_path, bbox_inches='tight', dpi=150)
    plt.show()
    print(f'Saved → {out_path}')
